# V5 inductive split seed check

Minimal diagnostic notebook.

Question 1: are inductive split seeds `42`, `43`, `44` systematically different in difficulty?

The only main plot below shows mean and median model performance per seed for `AUROC`, `AUPRC`, `MCC`, and `F1`. If one seed is globally harder, it should be lower across several metrics.

Question 2: is there a molecule that looks similar to many other molecules using receptor responses only?

The last plot builds molecule response profiles from a molecule-receptor pair table and ranks molecules by response-profile hubness.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

plt.rcParams.update({
    "figure.figsize": (11, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


In [ ]:
def find_repo_root() -> Path:
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (base / "results" / "graph" / "full_full" / "v5").exists():
            return base
        if (base / ".git").exists() and (base / "data").exists():
            return base
    raise FileNotFoundError("Could not find repository root")

ROOT = find_repo_root()
ARCH_TABLE_DIR = ROOT / "results" / "graph" / "full_full" / "v5" / "architecture_screen" / "reports" / "tables"
DATA_DIR = ROOT / "data" / "processed"

print("ROOT:", ROOT)


In [ ]:
# Use "best" for best-validation checkpoints, or "last" for last checkpoints.
CHECKPOINT_SOURCE = "best"

runs_path = {
    "best": ARCH_TABLE_DIR / "architecture_best_checkpoint_runs.csv",
    "last": ARCH_TABLE_DIR / "architecture_runs.csv",
}[CHECKPOINT_SOURCE]

metrics = ["AUROC", "AUPRC", "MCC", "F1"]

runs = pd.read_csv(runs_path)
ind = runs[runs["regime"].eq("inductive_molecule")].copy()
ind["seed"] = ind["gnn_seed"].astype(int)
ind["method"] = ind["arch"].astype(str).str.upper() + " / " + ind["mp_mode"].astype(str)
for m in metrics:
    ind[m] = pd.to_numeric(ind[m], errors="coerce")

seed_summary = (
    ind.melt(id_vars=["seed", "method"], value_vars=metrics, var_name="metric", value_name="score")
       .dropna(subset=["score"])
       .groupby(["metric", "seed"], as_index=False)
       .agg(mean=("score", "mean"), median=("score", "median"), n_methods=("method", "nunique"))
)

print(f"Loaded {len(ind)} inductive runs from {runs_path.name}")
seed_summary


In [ ]:
# One compact diagnostic figure: mean and median by seed, one panel per metric.
fig, axes = plt.subplots(1, len(metrics), figsize=(4.2 * len(metrics), 4), sharey=False)
if len(metrics) == 1:
    axes = [axes]

for ax, metric in zip(axes, metrics):
    d = seed_summary[seed_summary["metric"].eq(metric)].sort_values("seed")
    x = np.arange(len(d))
    labels = d["seed"].astype(str).tolist()

    ax.bar(x, d["mean"], width=0.65, color="#4C78A8", alpha=0.85, label="mean")
    ax.scatter(x, d["median"], color="#F58518", s=90, zorder=3, label="median")

    for xi, mean, median in zip(x, d["mean"], d["median"]):
        ax.plot([xi, xi], [mean, median], color="#F58518", alpha=0.7, linewidth=2)

    y_min = max(0, min(d["mean"].min(), d["median"].min()) - 0.06)
    y_max = min(1, max(d["mean"].max(), d["median"].max()) + 0.06)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_title(metric)
    ax.set_xlabel("seed")
    ax.set_ylabel("score")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
fig.suptitle(f"Inductive split difficulty proxy ({CHECKPOINT_SOURCE} checkpoints)", y=1.06)
fig.tight_layout()
plt.show()


## LoRaX full_full response-profile matrix

This section uses the LoRaX M2OR split files directly, not `pairs_curated.csv`.

Rows are molecules (`SMILES`), columns are receptors/proteins (`Protein sequence`). Cell colors mean:

- red: measured active / positive response;
- blue: measured inactive / negative response;
- white: no measurement for this molecule-protein pair.

The plot can work in two modes:

1. full mode: show all LoRaX molecules and all LoRaX proteins;
2. similar-only mode: show only molecules that have at least one other molecule with similarity above the threshold.

Set `SHOW_ONLY_SIMILAR_ROWS = True` and `SIMILARITY_THRESHOLD = 1.0` to show molecules whose response profiles repeat with at least one other molecule under the current weighted similarity scheme.

The similarity scheme assigns rewards and penalties to every pair of cell states:

| molecule A | molecule B | meaning | weight |
|---|---|---|---:|
| red | red | shared activation, strong evidence of similarity | `ACTIVE_ACTIVE_REWARD` |
| blue | blue | shared inactivity, weak evidence of similarity | `INACTIVE_INACTIVE_REWARD` |
| red | blue | direct contradiction, strong evidence of dissimilarity | `ACTIVE_INACTIVE_PENALTY` |
| red | missing | unresolved active signal, small penalty | `ACTIVE_MISSING_PENALTY` |
| blue | missing | weak/no evidence | `INACTIVE_MISSING_PENALTY` |
| missing | missing | no evidence | 0 |

`SIMILARITY_THRESHOLD` is used both for optional row filtering and for soft local ordering.


In [ ]:
# Load LoRaX full_full data directly from all provided random splits.
# This is the same molecule universe used by ChemBERTa LoRaX embeddings.
LORAX_DIR = ROOT / "data" / "external" / "lorax_m2or"
LORAX_SPLIT_GLOB = "rand_split_*/*.csv"

# Keep all molecules and proteins. These filters only remove completely uninformative molecules.
MIN_MEASURED_RECEPTORS = 1
MIN_ACTIVE_RECEPTORS = 0

lorax_files = sorted(LORAX_DIR.glob(LORAX_SPLIT_GLOB))
if not lorax_files:
    raise FileNotFoundError(f"No LoRaX split files found under {LORAX_DIR / LORAX_SPLIT_GLOB}")

frames = []
for path in lorax_files:
    df = pd.read_csv(path)
    df["split_file"] = str(path.relative_to(ROOT))
    frames.append(df)

pairs = pd.concat(frames, ignore_index=True)

required_cols = {"SMILES", "Protein sequence", "output"}
missing_cols = required_cols - set(pairs.columns)
if missing_cols:
    raise ValueError(f"LoRaX files are missing required columns: {sorted(missing_cols)}")

mol_col = "SMILES"
rec_col = "Protein sequence"
label_col = "output"

print("LoRaX files:", len(lorax_files))
print("LoRaX rows before pair deduplication:", len(pairs))
print("Using columns:", {"molecule": mol_col, "receptor": rec_col, "label": label_col})

work = pairs[[mol_col, rec_col, label_col]].copy()
work = work.dropna(subset=[mol_col, rec_col, label_col])
work[label_col] = pd.to_numeric(work[label_col], errors="coerce")
work = work.dropna(subset=[label_col])
work["response"] = np.where(work[label_col] > 0, 1, 0)

# Across repeated LoRaX split files, the same molecule-protein pair can appear multiple times.
# Use max aggregation: if any record is positive, the pair is positive.
# Missing pairs remain NaN; measured negatives are 0; positives are 1.
response = work.pivot_table(index=mol_col, columns=rec_col, values="response", aggfunc="max")

measured_count = response.notna().sum(axis=1)
active_count = response.fillna(0).sum(axis=1)
response = response.loc[(measured_count >= MIN_MEASURED_RECEPTORS) & (active_count >= MIN_ACTIVE_RECEPTORS)]

print("LoRaX unique molecules shown:", response.shape[0])
print("LoRaX unique proteins shown:", response.shape[1])
print("Mean measured proteins per molecule:", round(float(response.notna().sum(axis=1).mean()), 2))
print("Mean active proteins per molecule:", round(float(response.fillna(0).sum(axis=1).mean()), 2))


In [ ]:
# Order and optionally filter molecules.
#
# SHOW_ONLY_SIMILAR_ROWS = False:
#   keep all molecules and use SIMILARITY_THRESHOLD only for local ordering.
#
# SHOW_ONLY_SIMILAR_ROWS = True:
#   keep only molecules that have at least one other molecule with
#   weighted similarity >= SIMILARITY_THRESHOLD.
#   With SIMILARITY_THRESHOLD = 1.0 this highlights repeated response profiles
#   under the current weighted similarity scheme.
SIMILARITY_THRESHOLD = 0.9
SHOW_ONLY_SIMILAR_ROWS = True

# Pairwise state weights. Tune these if the biological interpretation changes.
ACTIVE_ACTIVE_REWARD = 1.00       # red-red: strong similarity
INACTIVE_INACTIVE_REWARD = 0.10   # blue-blue: weak similarity; do not let negatives dominate
ACTIVE_INACTIVE_PENALTY = 2.00    # red-blue: strong contradiction
ACTIVE_MISSING_PENALTY = 0.05     # red-missing: tiny uncertainty penalty
INACTIVE_MISSING_PENALTY = 0.00   # blue-missing: essentially no evidence

def compute_weighted_similarity(response_df):
    A = response_df.to_numpy(dtype=float)
    active = (A == 1).astype(np.float32)
    inactive = (A == 0).astype(np.float32)
    missing = np.isnan(A).astype(np.float32)

    aa = active @ active.T
    ii = inactive @ inactive.T
    ai = active @ inactive.T + inactive @ active.T
    am = active @ missing.T + missing @ active.T
    im = inactive @ missing.T + missing @ inactive.T

    both_measured = aa + ii + ai
    reward = ACTIVE_ACTIVE_REWARD * aa + INACTIVE_INACTIVE_REWARD * ii
    penalty = (
        ACTIVE_INACTIVE_PENALTY * ai
        + ACTIVE_MISSING_PENALTY * am
        + INACTIVE_MISSING_PENALTY * im
    )

    # Normalize by comparable evidence, not by reward+penalty only.
    denom = both_measured + ACTIVE_MISSING_PENALTY * am + INACTIVE_MISSING_PENALTY * im
    sim = np.divide(reward - penalty, denom, out=np.zeros_like(reward, dtype=float), where=denom > 0)
    np.fill_diagonal(sim, 0.0)
    return sim

full_similarity = compute_weighted_similarity(response)
max_similarity_to_other = full_similarity.max(axis=1) if response.shape[0] else np.array([])

if SHOW_ONLY_SIMILAR_ROWS:
    keep_mask = max_similarity_to_other >= SIMILARITY_THRESHOLD
    response_for_plot = response.loc[keep_mask].copy()
else:
    keep_mask = np.ones(response.shape[0], dtype=bool)
    response_for_plot = response.copy()

if response_for_plot.empty:
    raise ValueError(
        "No molecules passed the similarity filter. "
        "Try lowering SIMILARITY_THRESHOLD or set SHOW_ONLY_SIMILAR_ROWS = False."
    )

similarity = compute_weighted_similarity(response_for_plot)
measured_counts = response_for_plot.notna().sum(axis=1).to_numpy(dtype=float)

# Greedy nearest-neighbor order, starting from the most measured molecule.
# Coverage is the primary criterion; weighted similarity controls local grouping.
n = response_for_plot.shape[0]
remaining = set(range(n))
current = int(np.argmax(measured_counts))
order = [current]
remaining.remove(current)

def candidate_key(j, current):
    return (
        round(float(measured_counts[j]), 6),
        round(float(similarity[current, j]), 6),
        round(float(similarity[:, j].sum()), 6),
    )

threshold_hits = 0
threshold_fallbacks = 0
while remaining:
    above_threshold = [j for j in remaining if similarity[current, j] >= SIMILARITY_THRESHOLD]
    if above_threshold:
        candidates = above_threshold
        threshold_hits += 1
    else:
        candidates = list(remaining)
        threshold_fallbacks += 1

    next_i = max(candidates, key=lambda j: candidate_key(j, current))
    order.append(next_i)
    remaining.remove(next_i)
    current = next_i

clustered_response = response_for_plot.iloc[order]
row_active_count = clustered_response.fillna(0).sum(axis=1).astype(int)
row_measured_count = clustered_response.notna().sum(axis=1).astype(int)

# Order receptors by how many measurements/positives they have, for a denser left side.
col_order = pd.DataFrame({
    "measured": clustered_response.notna().sum(axis=0),
    "active": clustered_response.fillna(0).sum(axis=0),
}).sort_values(["measured", "active"], ascending=False).index
clustered_response = clustered_response[col_order]

print("Similarity threshold:", SIMILARITY_THRESHOLD)
print("Show only similar rows:", SHOW_ONLY_SIMILAR_ROWS)
print("All molecules before optional similarity filter:", response.shape[0])
print("Molecules shown:", clustered_response.shape[0])
print("Receptors shown:", clustered_response.shape[1])
print("Molecules with at least one neighbor above threshold:", int((max_similarity_to_other >= SIMILARITY_THRESHOLD).sum()))
print("Threshold-guided row transitions:", threshold_hits)
print("Fallback row transitions:", threshold_fallbacks)
print("Full-matrix similarity range:", round(float(np.nanmin(full_similarity)), 3), "to", round(float(np.nanmax(full_similarity)), 3))
print("Top row measured receptor count:", int(row_measured_count.iloc[0]))
print("Bottom row measured receptor count:", int(row_measured_count.iloc[-1]))


In [ ]:
# Plot molecule x protein response matrix.
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

def short_receptor_name(x, max_len=16):
    # LoRaX receptor ids are protein sequences; show a compact sequence prefix only.
    s = str(x)
    return s[:max_len]

def short_molecule_name(x, max_len=22):
    return str(x)[:max_len]

# Encode missing=0, inactive=1, active=2.
encoded = clustered_response.copy()
encoded = encoded.where(encoded.notna(), -1)
encoded = encoded.replace({-1: 0, 0: 1, 1: 2}).to_numpy(dtype=int)

# white = missing, blue = measured inactive, red = measured active
cmap = ListedColormap(["#FFFFFF", "#4C78A8", "#D62728"])
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap.N)

n_mols, n_receptors = clustered_response.shape
fig_w = max(14, 0.11 * n_receptors + 6)
fig_h = max(10, 0.035 * n_mols + 5)
fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=160)
im = ax.imshow(encoded, aspect="auto", interpolation="nearest", cmap=cmap, norm=norm)

# Molecule labels: sparse if there are many rows. The matrix still contains all rows.
y_step = max(1, n_mols // 60)
yticks = np.arange(0, n_mols, y_step)
row_labels = [
    f"{short_molecule_name(mol)}  m={m} a={a}"
    for mol, m, a in zip(clustered_response.index[yticks], row_measured_count.iloc[yticks], row_active_count.iloc[yticks])
]
ax.set_yticks(yticks)
ax.set_yticklabels(row_labels, fontsize=6)

# Protein labels: short sequence prefixes only, sparse if needed. The matrix still contains all columns.
x_step = max(1, n_receptors // 80)
xticks = np.arange(0, n_receptors, x_step)
ax.set_xticks(xticks)
ax.set_xticklabels([short_receptor_name(c) for c in clustered_response.columns[xticks]], rotation=90, fontsize=6)

ax.set_xlabel("protein sequence")
ax.set_ylabel("molecule SMILES, ordered by measured coverage and weighted response similarity")
ax.set_title(f"LoRaX molecule x protein response matrix: {n_mols} molecules x {n_receptors} proteins")

legend = [
    Patch(facecolor="#D62728", edgecolor="black", label="active / positive"),
    Patch(facecolor="#4C78A8", edgecolor="black", label="inactive / negative"),
    Patch(facecolor="#FFFFFF", edgecolor="black", label="missing"),
]
ax.legend(handles=legend, loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False)

fig.tight_layout()
plt.show()


### ChemBERTa distances for response-profile-similar molecule pairs

This cell is meant to be read together with the LoRaX response-profile matrix above.

It takes molecule pairs with weighted response similarity `>= SIMILARITY_THRESHOLD`, then checks their ChemBERTa cosine distances after mean-centering the embedding space.

Use case:

- set `SHOW_ONLY_SIMILAR_ROWS = True`;
- set an aggressive `SIMILARITY_THRESHOLD`, for example `1.0`;
- rerun the response-profile cells and this cell;
- inspect whether LoRaX response-profile similarity corresponds to ChemBERTa closeness.


In [ ]:
# ChemBERTa cosine-distance distribution for molecule pairs similar by response profile.
# Uses the same SIMILARITY_THRESHOLD and full_similarity computed above.
CHEMBERTA_CANDIDATES = [
    ROOT / "data" / "embeddings" / "molecules" / "chemberta_77m_lorax_std.pkl",
    ROOT / "data" / "embeddings" / "molecules" / "chemberta_77m_lorax.pkl",
]

chemberta_path = next((p for p in CHEMBERTA_CANDIDATES if p.exists()), None)
if chemberta_path is None:
    raise FileNotFoundError("No ChemBERTa embedding file found. Tried: " + ", ".join(map(str, CHEMBERTA_CANDIDATES)))

def extract_embedding_matrix_with_ids(obj):
    if isinstance(obj, pd.DataFrame):
        numeric = obj.select_dtypes(include=[np.number])
        if numeric.shape[1] == 0:
            raise ValueError("DataFrame has no numeric embedding columns")

        id_candidates = [
            "molecule", "molecule_id", "mol_id", "inchi_key", "inchikey",
            "ligand", "odorant", "smiles", "cid", "name",
        ]
        id_col = None
        lower = {c.lower(): c for c in obj.columns}
        for cand in id_candidates:
            if cand.lower() in lower and lower[cand.lower()] not in numeric.columns:
                id_col = lower[cand.lower()]
                break

        ids = obj[id_col].astype(str) if id_col is not None else obj.index.astype(str)
        return numeric.to_numpy(dtype=float), pd.Index(ids.astype(str))

    if isinstance(obj, dict):
        ids = pd.Index([str(k) for k in obj.keys()])
        X = np.asarray(list(obj.values()), dtype=float)
        if X.ndim != 2:
            raise ValueError(f"Dict values do not form a 2D matrix; got shape {X.shape}")
        return X, ids

    X = np.asarray(obj, dtype=float)
    if X.ndim != 2:
        raise ValueError(f"Could not extract a 2D embedding matrix; got shape {X.shape}")
    return X, pd.Index([str(i) for i in range(X.shape[0])])

chemberta_obj = pd.read_pickle(chemberta_path)
X_chem, chem_ids = extract_embedding_matrix_with_ids(chemberta_obj)

finite_mask = np.isfinite(X_chem).all(axis=1)
X_chem = X_chem[finite_mask]
chem_ids = chem_ids[finite_mask]

# Mean-center using all available ChemBERTa vectors, then normalize.
X_centered = X_chem - X_chem.mean(axis=0, keepdims=True)
norms = np.linalg.norm(X_centered, axis=1)
nonzero = norms > 0
X_unit = X_centered[nonzero] / norms[nonzero, None]
chem_ids = chem_ids[nonzero]

# If duplicate ids exist, keep the first vector.
emb = pd.DataFrame(X_unit, index=chem_ids.astype(str))
emb = emb[~emb.index.duplicated(keep="first")]

# Build an id bridge. In this project response rows are usually InChIKey,
# while ChemBERTa embeddings are keyed by SMILES. String-matching raw SMILES is
# fragile, so the preferred bridge is: ChemBERTa SMILES -> RDKit InChIKey.
response_ids = pd.Index(response.index.astype(str))

id_bridge = pd.Series(response_ids, index=response_ids, dtype=object)  # response id -> embedding id
bridge_method = "identity"

try:
    from rdkit import Chem
    from rdkit.Chem import inchi

    chem_smiles_to_inchikey = {}
    failed_smiles = 0
    for emb_id in emb.index.astype(str):
        mol = Chem.MolFromSmiles(emb_id)
        if mol is None:
            failed_smiles += 1
            continue
        ik = inchi.MolToInchiKey(mol)
        if ik and ik not in chem_smiles_to_inchikey:
            chem_smiles_to_inchikey[ik] = emb_id

    rdkit_bridge = pd.Series(chem_smiles_to_inchikey, dtype=object)  # inchikey -> chemberta smiles key
    id_bridge = pd.Series(response_ids, index=response_ids, dtype=object).map(rdkit_bridge).fillna(pd.Series(response_ids, index=response_ids, dtype=object))
    bridge_method = f"RDKit SMILES->InChIKey ({len(rdkit_bridge)} parsed, {failed_smiles} failed)"
except Exception as e:
    print("RDKit bridge unavailable; falling back to raw SMILES string mapping:", e)

    if "smiles" in pairs.columns and mol_col in pairs.columns:
        bridge_df = pairs[[mol_col, "smiles"]].dropna().drop_duplicates(subset=[mol_col])
        bridge = pd.Series(bridge_df["smiles"].astype(str).values, index=bridge_df[mol_col].astype(str).values)
        id_bridge = pd.Series(response_ids, index=response_ids, dtype=object).map(bridge).fillna(pd.Series(response_ids, index=response_ids, dtype=object))
        bridge_method = "raw pairs smiles"

    molecule_smiles_path = DATA_DIR / "molecules" / "molecule_smiles.csv"
    if molecule_smiles_path.exists():
        mol_smiles = pd.read_csv(molecule_smiles_path)
        if {"inchikey", "smiles"}.issubset(mol_smiles.columns):
            bridge = pd.Series(mol_smiles["smiles"].astype(str).values, index=mol_smiles["inchikey"].astype(str).values)
            unresolved = ~id_bridge.isin(emb.index)
            if unresolved.any():
                remapped = pd.Series(response_ids, index=response_ids, dtype=object).map(bridge)
                id_bridge.loc[unresolved] = remapped.loc[unresolved].fillna(id_bridge.loc[unresolved])

common_response_ids = response_ids[id_bridge.reindex(response_ids).isin(emb.index).to_numpy()]
if len(common_response_ids) == 0:
    print("Bridge method:", bridge_method)
    print("Response id sample:", response_ids[:5].tolist())
    print("Mapped embedding id sample:", id_bridge.reindex(response_ids).head().tolist())
    print("ChemBERTa id sample:", emb.index[:5].tolist())
    raise ValueError(
        "No overlap between response molecule ids and ChemBERTa ids after id mapping. "
        "Inspect response ids, mapped ids, and ChemBERTa ids above."
    )

# Similar pairs in the full response matrix, not only currently shown rows.
tri_i, tri_j = np.triu_indices(response.shape[0], k=1)
sim_values = full_similarity[tri_i, tri_j]
sim_pair_mask = sim_values >= SIMILARITY_THRESHOLD

response_id_array = response.index.astype(str).to_numpy()
pair_left = response_id_array[tri_i[sim_pair_mask]]
pair_right = response_id_array[tri_j[sim_pair_mask]]
pair_sim = sim_values[sim_pair_mask]

pairs_df = pd.DataFrame({
    "mol_a": pair_left,
    "mol_b": pair_right,
    "response_similarity": pair_sim,
})

pairs_df["emb_id_a"] = pairs_df["mol_a"].map(id_bridge)
pairs_df["emb_id_b"] = pairs_df["mol_b"].map(id_bridge)

# Keep only pairs where both molecules have ChemBERTa embeddings.
pairs_df = pairs_df[pairs_df["emb_id_a"].isin(emb.index) & pairs_df["emb_id_b"].isin(emb.index)].copy()
if pairs_df.empty:
    print("Mapped response ids with ChemBERTa:", len(common_response_ids))
    print("Example mapped ids:", id_bridge.reindex(response_ids).head().tolist())
    raise ValueError(
        "No response-similar molecule pairs with ChemBERTa embeddings. "
        "Try lowering SIMILARITY_THRESHOLD or inspect id mapping."
    )

A = emb.loc[pairs_df["emb_id_a"]].to_numpy(dtype=float)
B = emb.loc[pairs_df["emb_id_b"]].to_numpy(dtype=float)
cos_sim = np.sum(A * B, axis=1)
pairs_df["chemberta_cosine_distance"] = 1.0 - cos_sim

print("ChemBERTa file:", chemberta_path.relative_to(ROOT))
print("ID bridge method:", bridge_method)
print("Response molecules:", response.shape[0])
print("ChemBERTa vectors:", emb.shape[0])
print("Response molecules mapped to ChemBERTa:", len(common_response_ids))
print("Response-similar pairs at threshold:", int(sim_pair_mask.sum()))
print("Response-similar pairs with ChemBERTa:", len(pairs_df))
print("ChemBERTa cosine distance min / median / mean / max:",
      round(float(pairs_df["chemberta_cosine_distance"].min()), 4),
      round(float(pairs_df["chemberta_cosine_distance"].median()), 4),
      round(float(pairs_df["chemberta_cosine_distance"].mean()), 4),
      round(float(pairs_df["chemberta_cosine_distance"].max()), 4))

display_cols = ["mol_a", "mol_b", "response_similarity", "chemberta_cosine_distance", "emb_id_a", "emb_id_b"]
display(pairs_df.sort_values(["response_similarity", "chemberta_cosine_distance"], ascending=[False, True])[display_cols].head(20))

fig, ax = plt.subplots(figsize=(8, 4.5))
d = pairs_df["chemberta_cosine_distance"].to_numpy(dtype=float)

try:
    from scipy.stats import gaussian_kde
    xs = np.linspace(float(d.min()), float(d.max()), 512)
    kde = gaussian_kde(d)
    ax.plot(xs, kde(xs), color="#4C78A8", linewidth=2.2, label="KDE")
    ax.fill_between(xs, kde(xs), color="#4C78A8", alpha=0.2)
except Exception as e:
    print("scipy KDE unavailable; using histogram density only:", e)

ax.hist(d, bins=min(80, max(10, int(np.sqrt(len(d))))), density=True, color="#A0CBE8", alpha=0.5, edgecolor="none", label="hist density")
ax.axvline(np.mean(d), color="#D62728", linestyle="--", linewidth=1.8, label="mean")
ax.axvline(np.median(d), color="#F58518", linestyle=":", linewidth=2.0, label="median")
ax.set_title(f"ChemBERTa distances for response-similar pairs (threshold={SIMILARITY_THRESHOLD})")
ax.set_xlabel("centered ChemBERTa cosine distance")
ax.set_ylabel("density")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


Notes:

- The heatmap now uses LoRaX split files directly, so its molecule universe should match LoRaX ChemBERTa embeddings.
- `SHOW_ONLY_SIMILAR_ROWS` is the separate filtering mode.
- If `SHOW_ONLY_SIMILAR_ROWS = True`, a molecule is shown only when it has at least one other molecule with similarity >= `SIMILARITY_THRESHOLD`.
- With `SIMILARITY_THRESHOLD = 1.0`, this shows response-profile repeats under the current weighted scoring scheme.
- Dense molecules are placed near the top; each y-label includes `m=` measured proteins and `a=` active proteins.
- Missing measurements are not treated as negatives unless you change the constants above.
- Protein labels are shortened to sequence prefixes only for display.


## ChemBERTa centered embedding cosine distances

This final check uses ChemBERTa molecule embeddings only.

Steps:

1. load ChemBERTa vectors;
2. subtract the global mean vector from every molecule vector;
3. compute pairwise cosine distances;
4. plot the density of all off-diagonal pairwise distances.

This helps inspect whether the embedding space has a broad continuum or suspicious concentration/collapse after centering.


In [ ]:
# Center ChemBERTa vectors and inspect pairwise cosine-distance distribution.
CHEMBERTA_CANDIDATES = [
    ROOT / "data" / "embeddings" / "molecules" / "chemberta_77m_lorax_std.pkl",
    ROOT / "data" / "embeddings" / "molecules" / "chemberta_77m_lorax.pkl",
]

chemberta_path = next((p for p in CHEMBERTA_CANDIDATES if p.exists()), None)
if chemberta_path is None:
    raise FileNotFoundError("No ChemBERTa embedding file found. Tried: " + ", ".join(map(str, CHEMBERTA_CANDIDATES)))

obj = pd.read_pickle(chemberta_path)
print("ChemBERTa file:", chemberta_path.relative_to(ROOT))
print("Loaded object type:", type(obj))

def extract_embedding_matrix(obj):
    # Common case: DataFrame with one id column and many numeric embedding columns.
    if isinstance(obj, pd.DataFrame):
        numeric = obj.select_dtypes(include=[np.number])
        if numeric.shape[1] == 0:
            raise ValueError("DataFrame has no numeric embedding columns")
        ids = obj.index.astype(str)
        return numeric.to_numpy(dtype=float), ids

    # Common case: dict molecule_id -> vector.
    if isinstance(obj, dict):
        ids = list(obj.keys())
        values = list(obj.values())
        X = np.asarray(values, dtype=float)
        if X.ndim != 2:
            raise ValueError(f"Dict values do not form a 2D matrix; got shape {X.shape}")
        return X, pd.Index([str(x) for x in ids])

    # Common case: plain numpy-like matrix.
    X = np.asarray(obj, dtype=float)
    if X.ndim != 2:
        raise ValueError(f"Could not extract a 2D embedding matrix; got shape {X.shape}")
    return X, pd.Index([str(i) for i in range(X.shape[0])])

X, chemberta_ids = extract_embedding_matrix(obj)
X = np.asarray(X, dtype=float)

# Remove rows with non-finite values just in case.
finite_mask = np.isfinite(X).all(axis=1)
if not finite_mask.all():
    print("Dropping non-finite vectors:", int((~finite_mask).sum()))
    X = X[finite_mask]
    chemberta_ids = chemberta_ids[finite_mask]

X_centered = X - X.mean(axis=0, keepdims=True)
norms = np.linalg.norm(X_centered, axis=1)
nonzero = norms > 0
if not nonzero.all():
    print("Dropping zero-norm centered vectors:", int((~nonzero).sum()))
    X_centered = X_centered[nonzero]
    chemberta_ids = chemberta_ids[nonzero]
    norms = norms[nonzero]

X_unit = X_centered / norms[:, None]
cosine_similarity = X_unit @ X_unit.T
cosine_distance = 1.0 - cosine_similarity

# Off-diagonal upper triangle only.
tri_i, tri_j = np.triu_indices(cosine_distance.shape[0], k=1)
distances = cosine_distance[tri_i, tri_j]

print("Vectors:", X_unit.shape[0])
print("Dimensions:", X_unit.shape[1])
print("Pairwise distances:", distances.size)
print("Distance min / median / mean / max:",
      round(float(np.min(distances)), 4),
      round(float(np.median(distances)), 4),
      round(float(np.mean(distances)), 4),
      round(float(np.max(distances)), 4))


In [ ]:
# Plot density of centered ChemBERTa pairwise cosine distances.
fig, ax = plt.subplots(figsize=(8, 4.5))

try:
    from scipy.stats import gaussian_kde
    xs = np.linspace(float(distances.min()), float(distances.max()), 512)
    kde = gaussian_kde(distances)
    ax.plot(xs, kde(xs), color="#4C78A8", linewidth=2.2, label="KDE")
    ax.fill_between(xs, kde(xs), color="#4C78A8", alpha=0.2)
except Exception as e:
    print("scipy KDE unavailable; using histogram density instead:", e)

ax.hist(distances, bins=80, density=True, color="#A0CBE8", alpha=0.45, edgecolor="none", label="hist density")
ax.axvline(np.mean(distances), color="#D62728", linestyle="--", linewidth=1.8, label="mean")
ax.axvline(np.median(distances), color="#F58518", linestyle=":", linewidth=2.0, label="median")

ax.set_title("Centered ChemBERTa embeddings: pairwise cosine-distance density")
ax.set_xlabel("cosine distance after mean-centering")
ax.set_ylabel("density")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()
